In [1]:
import pandas as pd
import requests
import io
import os
from datetime import datetime


# paths
save_dir = (
    "/lakehouse/default/Files/data/raw/"
    "bse_security_inc"
)

log_file = (
    "/lakehouse/default/Files/data/raw/"
    "logs/pipeline_log.csv"
)

os.makedirs(save_dir, exist_ok=True)

# today
today = datetime.today()

file_date = today.strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"bse_security_{file_date}.csv"
)

try:

    url = (
        "https://api.bseindia.com/BseIndiaAPI/api/"
        "LitsOfScripCSVDownload/w"
        "?Group="
        "&Scripcode="
        "&segment=Equity"
        "&status=Active"
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    # keep only real columns
    df = pd.read_csv(
        io.StringIO(response.text),
        usecols=range(9)
    )

    # clean columns
    df.columns = (
        df.columns
        .str.strip()
    )

    # save snapshot
    df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(df)
    message = "File saved"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)


# logging
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "bse_security",
    "status": status,
    "rows": rows,
    "message": message
}])

if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row


log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, c3595aae-c8f5-4279-baa0-f83d07a8e122, 3, Finished, Available, Finished, False)

SUCCESS
Rows: 4873
File saved


In [2]:
print("montly update completed")

StatementMeta(, c3595aae-c8f5-4279-baa0-f83d07a8e122, 4, Finished, Available, Finished, False)

montly update completed
